In [1]:
# Module 5: Context Management & Multi-Agent Orchestration
# Lab: Research & Synthesis Pipeline

# Setup -- install dependencies (run once per session)
# !pip install -q claude-agent-sdk python-dotenv

In [2]:
# Import libraries
import os
import json
import asyncio
from pathlib import Path
from dotenv import load_dotenv
from claude_agent_sdk import query, ClaudeAgentOptions, HookMatcher


In [3]:
# Load API keys from .env file
load_dotenv()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
print(f"Anthropic key (SDK): {'Yes' if ANTHROPIC_API_KEY else 'No'}")

Anthropic key (SDK): Yes


In [4]:
# Step 1 -- Define the Researcher sub-agent
# The Researcher only has WebSearch and WebFetch. No write access.
async def run_researcher(topic: str) -> str:
    """Gather research on a topic using web tools only."""
    options = ClaudeAgentOptions(
        allowed_tools=["WebSearch", "WebFetch"],
        model="claude-haiku-4-5-20251001",
    )
    prompt = f"""Research the topic '{topic}' and return concise findings.
You MUST call WebSearch first to find relevant information, then WebFetch to read details.
Return a bullet-point summary of the most important facts only."""
    result = ""
    async for message in query(prompt=prompt, options=options):
        if hasattr(message, 'content') and message.content:
            result = message.content
        if hasattr(message, 'result') and message.result:
            result = message.result
    return result

In [5]:
# Step 2 -- Define the Writer sub-agent
# The Writer only has the Edit tool. No web access.
async def can_use_tool(tool_name: str, input_data: dict, context):
    if tool_name == "Edit":
        file_path = input_data.get('file_path', 'unknown')
        response = input(f"Allow Edit on {file_path}? (y/n): ")
        if response.lower() == 'y':
            return {"behavior": "allow", "updatedInput": input_data}
        return {"behavior": "deny"}
    return {"behavior": "allow", "updatedInput": input_data}

async def run_writer(findings: str, template_path: str, output_path: str) -> str:
    """Write findings into the report template using Edit tool only."""
    options = ClaudeAgentOptions(
        allowed_tools=["Read", "Edit"],
        permission_mode="default",
        can_use_tool=can_use_tool,
        model="claude-haiku-4-5-20251001",
    )
    prompt = f"""Read the template at {template_path}, then write a completed
report to {output_path} using the Edit tool.

Findings to incorporate:
{findings}

Replace every placeholder in the template with real content.
Do NOT modify any other files."""
    result = ""
    async def prompt_stream():
        yield {
            "type": "user",
            "message": {"role": "user", "content": prompt},
            "parent_tool_use_id": None,
            "session_id": "",
        }

    async for message in query(prompt=prompt_stream(), options=options):
        if hasattr(message, 'content') and message.content:
            result = message.content
        if hasattr(message, 'result') and message.result:
            result = message.result
    return result

In [6]:
# Step 3 -- Define the Coordinator
# The Coordinator orchestrates the full pipeline: research -> write
async def run_coordinator(task: str, template_path: str, output_path: str) -> str:
    """Orchestrate research and writing phases."""
    print("[Coordinator] Starting research phase...")
    findings = await run_researcher(task)
    print(f"[Coordinator] Research complete. {len(findings)} chars gathered.")

    print("[Coordinator] Starting writing phase...")
    report = await run_writer(findings, template_path, output_path)
    print("[Coordinator] Report written.")

    return report

In [7]:
# Step 4 -- Execute the pipeline
# Set target paths and run the full orchestration
# Each sub-agent gets its own fresh context window
TEMPLATE_PATH = "data/report_template.md"
OUTPUT_PATH = "data/completed_report.md"
TASK = "Quantum Computing"

result = await run_coordinator(TASK, TEMPLATE_PATH, OUTPUT_PATH)
print("\n--- Final Report ---\n")
print(result)

[Coordinator] Starting research phase...
[Coordinator] Research complete. 2641 chars gathered.
[Coordinator] Starting writing phase...
[Coordinator] Report written.

--- Final Report ---

Perfect! I've successfully created the completed report at `/data/completed_report.md`. The report includes:

✅ **All placeholders replaced with real content:**
- **Topic**: "Quantum Computing Breakthroughs and Market Emergence in 2026"
- **Summary**: Comprehensive overview of 2026's quantum computing inflection point
- **Key Findings**: 8 detailed bullet points covering major breakthroughs (Google Willow, Microsoft Majorana 1), hardware diversity, error correction advances, QaaS platforms, pilot applications, investment trends, and cryptographic urgency
- **Implications**: Detailed analysis of what these findings mean for technology leaders, enterprise strategy, workforce development, market timing, and risk management
- **Sources**: All 5 academic and industry sources from your research summary

The

In [8]:
# Step 5 -- Verify the output
# Read the completed report to verify the Writer filled in the template
report_file = Path(OUTPUT_PATH)
if report_file.exists():
    print("--- Completed Report ---")
    print(report_file.read_text())
else:
    print("Report not found.")

--- Completed Report ---
# Research Report: Quantum Computing Breakthroughs and Market Emergence in 2026

## Summary
Quantum computing has reached a critical inflection point in 2026, marked by major breakthroughs in error correction and hardware diversity. Google's Willow chip achieved "below-threshold" error correction in December 2024, proving that quantum systems can scale reliably with fewer errors in larger systems. Microsoft introduced topological qubits with potential to scale to one million qubits on a single chip. The industry is transitioning from the Noisy Intermediate-Scale Quantum (NISQ) era toward fault-tolerant, stable architectures. With cloud-based quantum-as-a-service platforms eliminating capital barriers and commercial viability projected for the early 2030s, 2026 marks the "beginning of quantum industrialization."

## Key Findings
- **Google's Willow Chip** (December 2024) achieved "below-threshold" error correction—the first proof that quantum systems can scale r